# Inventory & Supply Chain Analytics (Maven Toys)

## Business Context
An e-commerce retailer sells toys across multiple stores and faces challenges related
to inventory imbalance. Overstocking results in cash being tied up in unsold products,
while stock-outs lead to lost sales and reduced customer satisfaction.

## Problem Statement
E-commerce businesses often struggle to balance inventory levels due to fluctuating
demand. This project aims to analyze sales and inventory data to identify demand–supply
mismatches, evaluate inventory health, and recommend actions to optimize inventory
management.

## Analytical Objectives
- Identify slow-moving and fast-moving products
- Detect products at risk of stock-outs
- Analyze demand variability across products
- Assess overall inventory health

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
products = pd.read_csv("products.csv")
stores = pd.read_csv("stores.csv")
sales = pd.read_csv("sales.csv")


In [11]:
# check dataset shapes

print("Products shape:", products.shape)
print("Stores shape:", stores.shape)
print("Sales shape:", sales.shape)

Products shape: (35, 5)
Stores shape: (50, 5)
Sales shape: (829262, 5)


In [12]:
# Column overview and data types

print("=== PRODUCTS COLUMNS & TYPES ===")
print(products.dtypes)
print("\n")

print("=== STORES COLUMNS & TYPES ===")
print(stores.dtypes)
print("\n")

print("=== SALES COLUMNS & TYPES ===")
print(sales.dtypes)

=== PRODUCTS COLUMNS & TYPES ===
Product_ID           int64
Product_Name        object
Product_Category    object
Product_Cost        object
Product_Price       object
dtype: object


=== STORES COLUMNS & TYPES ===
Store_ID            int64
Store_Name         object
Store_City         object
Store_Location     object
Store_Open_Date    object
dtype: object


=== SALES COLUMNS & TYPES ===
Sale_ID        int64
Date          object
Store_ID       int64
Product_ID     int64
Units          int64
dtype: object


In [13]:
#  Missing values check

print("=== MISSING VALUES ===")

print("\nProducts:")
print(products.isna().sum())

print("\nStores:")
print(stores.isna().sum())

print("\nSales:")
print(sales.isna().sum())

=== MISSING VALUES ===

Products:
Product_ID          0
Product_Name        0
Product_Category    0
Product_Cost        0
Product_Price       0
dtype: int64

Stores:
Store_ID           0
Store_Name         0
Store_City         0
Store_Location     0
Store_Open_Date    0
dtype: int64

Sales:
Sale_ID       0
Date          0
Store_ID      0
Product_ID    0
Units         0
dtype: int64


In [14]:
# Step 8: Duplicate checks

print("Duplicate rows in products:", products.duplicated().sum())
print("Duplicate rows in stores:", stores.duplicated().sum())
print("Duplicate rows in sales:", sales.duplicated().sum())

Duplicate rows in products: 0
Duplicate rows in stores: 0
Duplicate rows in sales: 0


##### DATA TYPE CONVERSIONS 

In [15]:
# Convert product cost and price to numeric

products["Product_Cost"] = (
    products["Product_Cost"]
    .str.replace("$", "", regex=False)
    .astype(float)
)

products["Product_Price"] = (
    products["Product_Price"]
    .str.replace("$", "", regex=False)
    .astype(float)
)

In [17]:
#  Convert date columns to datetime

sales["Date"] = pd.to_datetime(sales["Date"])
stores["Store_Open_Date"] = pd.to_datetime(stores["Store_Open_Date"])

In [18]:
sales.dtypes, stores.dtypes

(Sale_ID                int64
 Date          datetime64[ns]
 Store_ID               int64
 Product_ID             int64
 Units                  int64
 dtype: object,
 Store_ID                    int64
 Store_Name                 object
 Store_City                 object
 Store_Location             object
 Store_Open_Date    datetime64[ns]
 dtype: object)

#### FEATURE ENGINEERING

In [19]:
#  Time-based features from sales date

sales["Year"] = sales["Date"].dt.year
sales["Month"] = sales["Date"].dt.month
sales["Month_Name"] = sales["Date"].dt.month_name()

In [22]:
#  Merge sales with product price

sales_enriched = sales.merge(
    products[["Product_ID", "Product_Price"]],
    on="Product_ID",
    how="left"
)

#  Calculate revenue

sales_enriched["Revenue"] = (
    sales_enriched["Units"] * sales_enriched["Product_Price"]
)

In [23]:
sales_enriched.head()

,Sale_ID,Date,Store_ID,Product_ID,Units,Year,Month,Month_Name,Product_Price,Revenue
0,1,2022-01-01,24,4,1,2022,1,January,12.99,12.99
1,2,2022-01-01,28,1,1,2022,1,January,15.99,15.99
2,3,2022-01-01,6,8,1,2022,1,January,6.99,6.99
3,4,2022-01-01,48,7,1,2022,1,January,15.99,15.99
4,5,2022-01-01,44,18,1,2022,1,January,39.99,39.99


#### AGGREGATIONS (PRODUCT & TIME LEVEL)

In [24]:
#  Product-level aggregation

product_performance = (
    sales_enriched
    .groupby("Product_ID")
    .agg(
        Total_Units_Sold=("Units", "sum"),
        Total_Revenue=("Revenue", "sum"),
        Avg_Units_Per_Transaction=("Units", "mean"),
        Transaction_Count=("Sale_ID", "count")
    )
    .reset_index()
)

In [25]:
# Add product names for readability

product_performance = product_performance.merge(
    products[["Product_ID", "Product_Name", "Product_Category"]],
    on="Product_ID",
    how="left"
)

In [26]:
#  Monthly sales trend

monthly_trend = (
    sales_enriched
    .groupby(["Year", "Month", "Month_Name"])
    .agg(
        Monthly_Units_Sold=("Units", "sum"),
        Monthly_Revenue=("Revenue", "sum")
    )
    .reset_index()
    .sort_values(["Year", "Month"])
)

In [27]:
product_performance.head()

,Product_ID,Total_Units_Sold,Total_Revenue,Avg_Units_Per_Transaction,Transaction_Count,Product_Name,Product_Category
0,1,57958,926748.42,1.195084,48497,Action Figure,Toys
1,2,39089,507766.11,1.212062,32250,Animal Figures,Toys
2,3,91663,365735.37,1.695015,54078,Barrel O' Slime,Art & Crafts
3,4,3829,49738.71,1.034865,3700,Chutes & Ladders,Games
4,5,4471,44665.29,1.047318,4269,Classic Dominoes,Games


In [28]:
monthly_trend.head()

,Year,Month,Month_Name,Monthly_Units_Sold,Monthly_Revenue
0,2022,1,January,38009,542554.91
1,2022,2,February,36935,541351.65
2,2022,3,March,39981,589485.19
3,2022,4,April,47102,681072.98
4,2022,5,May,46910,672369.90


##### FAST-MOVING vs SLOW-MOVING PRODUCTS

In [29]:
# Calculate overall average units sold per product

average_units_sold = product_performance["Total_Units_Sold"].mean()

average_units_sold

np.float64(31159.0)

In [30]:
#  Classify products as fast or slow moving

product_performance["Movement_Category"] = np.where(
    product_performance["Total_Units_Sold"] >= average_units_sold,
    "Fast-Moving",
    "Slow-Moving"
)

In [31]:
product_performance[["Product_Name", "Total_Units_Sold", "Movement_Category"]].head()

,Product_Name,Total_Units_Sold,Movement_Category
0,Action Figure,57958,Fast-Moving
1,Animal Figures,39089,Fast-Moving
2,Barrel O' Slime,91663,Fast-Moving
3,Chutes & Ladders,3829,Slow-Moving
4,Classic Dominoes,4471,Slow-Moving


#### REVENUE CONCENTRATION (PARETO ANALYSIS)

In [32]:
#  Sort products by total revenue

product_performance_sorted = (
    product_performance
    .sort_values("Total_Revenue", ascending=False)
    .reset_index(drop=True)
)

In [33]:
#  Calculate cumulative revenue contribution

product_performance_sorted["Cumulative_Revenue"] = (
    product_performance_sorted["Total_Revenue"].cumsum()
)

total_revenue = product_performance_sorted["Total_Revenue"].sum()

product_performance_sorted["Cumulative_Revenue_Percent"] = (
    product_performance_sorted["Cumulative_Revenue"] / total_revenue * 100
)

In [34]:
#  Products contributing to first 80% of revenue

top_revenue_products = product_performance_sorted[
    product_performance_sorted["Cumulative_Revenue_Percent"] <= 80
]

top_revenue_products.shape

(14, 10)

In [35]:
top_revenue_products[
    ["Product_Name", "Total_Revenue", "Cumulative_Revenue_Percent"]
].head()

,Product_Name,Total_Revenue,Cumulative_Revenue_Percent
0,Lego Bricks,2388882.63,16.538272
1,Colorbuds,1564476.32,27.369166
2,Magic Sand,968962.02,34.077305
3,Action Figure,926748.42,40.493199
4,Rubik's Cube,912983.28,46.813796


#### TREND ANALYSIS (MONTHLY DEMAND & REVENUE)

In [36]:
#  Review monthly trend data

monthly_trend.head()

,Year,Month,Month_Name,Monthly_Units_Sold,Monthly_Revenue
0,2022,1,January,38009,542554.91
1,2022,2,February,36935,541351.65
2,2022,3,March,39981,589485.19
3,2022,4,April,47102,681072.98
4,2022,5,May,46910,672369.90


In [37]:
# Month-over-month growth calculations

monthly_trend["MoM_Units_Change"] = monthly_trend["Monthly_Units_Sold"].pct_change() * 100
monthly_trend["MoM_Revenue_Change"] = monthly_trend["Monthly_Revenue"].pct_change() * 100

monthly_trend.head()

,Year,Month,Month_Name,Monthly_Units_Sold,Monthly_Revenue,MoM_Units_Change,MoM_Revenue_Change
0,2022,1,January,38009,542554.91,NaN,NaN
1,2022,2,February,36935,541351.65,-2.825647,-0.221777
2,2022,3,March,39981,589485.19,8.246920,8.891363
3,2022,4,April,47102,681072.98,17.810960,15.536911
4,2022,5,May,46910,672369.90,-0.407626,-1.277848


In [38]:
#  High-level trend summary

monthly_trend.describe()

,Year,Month,Monthly_Units_Sold,Monthly_Revenue,MoM_Units_Change,MoM_Revenue_Change
count,21.000000,21.000000,21.000000,21.000000,20.000000,20.000000
mean,2022.428571,5.857143,51931.666667,687836.778571,2.296675,1.779204
std,0.507093,3.275450,10171.922490,117969.145824,12.464249,13.334645
min,2022.000000,1.000000,36935.000000,489422.730000,-19.525576,-20.217543
25%,2022.000000,3.000000,46177.000000,589485.190000,-2.958942,-4.045235
50%,2022.000000,6.000000,51185.000000,661980.220000,0.081840,-0.346222
75%,2023.000000,8.000000,63614.000000,808299.250000,7.270564,7.091502
max,2023.000000,12.000000,69336.000000,883515.640000,29.004591,32.647540


#### STORE-LEVEL PERFORMANCE ANALYSIS

In [39]:
#  Store-level performance aggregation

store_performance = (
    sales_enriched
    .groupby("Store_ID")
    .agg(
        Total_Units_Sold=("Units", "sum"),
        Total_Revenue=("Revenue", "sum"),
        Transaction_Count=("Sale_ID", "count")
    )
    .reset_index()
)

In [40]:
#  Add store details

store_performance = store_performance.merge(
    stores[["Store_ID", "Store_Name", "Store_City", "Store_Location"]],
    on="Store_ID",
    how="left"
)

In [41]:
# Average revenue per transaction

store_performance["Avg_Revenue_Per_Transaction"] = (
    store_performance["Total_Revenue"] / store_performance["Transaction_Count"]
)

In [42]:
store_performance.head()

,Store_ID,Total_Units_Sold,Total_Revenue,Transaction_Count,Store_Name,Store_City,Store_Location,Avg_Revenue_Per_Transaction
0,1,20011,261842.89,15926,Maven Toys Guadalajara 1,Guadalajara,Residential,16.441221
1,2,20886,277959.14,15698,Maven Toys Monterrey 1,Monterrey,Residential,17.706659
2,3,20698,262435.02,16331,Maven Toys Guadalajara 2,Guadalajara,Commercial,16.069746
3,4,24010,330408.90,18924,Maven Toys Saltillo 1,Saltillo,Downtown,17.459781
4,5,16217,210897.83,13217,Maven Toys La Paz 1,La Paz,Downtown,15.956558


#### PRODUCT × STORE CROSS-ANALYSIS

In [43]:
# Product × Store performance

product_store_performance = (
    sales_enriched
    .groupby(["Store_ID", "Product_ID"])
    .agg(
        Units_Sold=("Units", "sum"),
        Revenue=("Revenue", "sum"),
        Transactions=("Sale_ID", "count")
    )
    .reset_index()
)

In [44]:
# Add product and store details

product_store_performance = (
    product_store_performance
    .merge(products[["Product_ID", "Product_Name", "Product_Category"]], 
           on="Product_ID", how="left")
    .merge(stores[["Store_ID", "Store_Name", "Store_City", "Store_Location"]], 
           on="Store_ID", how="left")
)

In [45]:
# Top product by units sold in each store

top_product_per_store = (
    product_store_performance
    .sort_values(["Store_ID", "Units_Sold"], ascending=[True, False])
    .groupby("Store_ID")
    .head(1)
    .reset_index(drop=True)
)

In [46]:
top_product_per_store[
    ["Store_Name", "Product_Name", "Units_Sold", "Revenue"]
].head()

,Store_Name,Product_Name,Units_Sold,Revenue
0,Maven Toys Guadalajara 1,Deck Of Cards,2124,14846.76
1,Maven Toys Monterrey 1,Barrel O' Slime,2614,10429.86
2,Maven Toys Guadalajara 2,Colorbuds,2629,39408.71
3,Maven Toys Saltillo 1,Colorbuds,2996,44910.04
4,Maven Toys La Paz 1,Deck Of Cards,2104,14706.96


## Key Insights

### 1. Demand Is Uneven Across Products
- A relatively small subset of products contributes a disproportionately large share of total revenue.
- This indicates a classic **Pareto pattern**, where focusing on top-performing SKUs can drive most business outcomes.

### 2. Presence of Slow-Moving Products
- Several products fall below the average units sold threshold, classifying them as slow-moving.
- These products are likely to tie up working capital without proportionate revenue contribution.

### 3. Revenue Concentration Risk
- Approximately 80% of total revenue is generated by a limited number of products.
- Heavy dependence on a small set of SKUs introduces revenue concentration risk if demand shifts.

### 4. Store Performance Varies Significantly
- Some stores consistently outperform others in both units sold and revenue.
- Differences in average revenue per transaction suggest variations in customer purchasing behavior across locations.

### 5. Store-Specific Product Preferences Exist
- Cross-analysis shows that top-selling products differ by store.
- This suggests that uniform inventory allocation across all stores may be inefficient.

## Business Recommendations

### 1. Prioritize Inventory for High-Revenue Products
- Focus inventory planning and replenishment on the small set of products contributing the majority of revenue.
- Ensure these SKUs have higher safety stock levels to reduce the risk of stock-outs.

**Expected Impact:**  
Improved revenue stability and reduced lost sales from high-demand products.

---

### 2. Rationalize Slow-Moving Inventory
- Review slow-moving products for potential actions such as promotions, bundling, or gradual phase-out.
- Avoid overstocking low-demand SKUs to free up working capital.

**Expected Impact:**  
Reduced inventory holding costs and better cash flow utilization.

---

### 3. Adopt Store-Specific Inventory Allocation
- Customize inventory allocation based on store-level demand patterns instead of using a uniform distribution strategy.
- Allocate higher quantities of store-specific top-selling products to relevant locations.

**Expected Impact:**  
Higher sell-through rates and improved store-level performance.

---

### 4. Monitor Revenue Concentration Risk
- While top products should be prioritized, actively monitor dependency on a small number of SKUs.
- Gradually diversify revenue sources by testing demand for mid-performing products.

**Expected Impact:**  
Lower business risk from demand shocks affecting a few key products.

---

### 5. Use Monthly Trend Monitoring for Planning
- Track month-over-month demand and revenue trends to anticipate periods of high or low sales activity.
- Align procurement and promotions with observed demand fluctuations.

**Expected Impact:**  
Better demand planning and more proactive inventory decisions.

## Validation, Assumptions & Limitations

### Data Validation
- Verified dataset dimensions and data types before analysis.
- Confirmed absence of missing values and duplicate records in core tables.
- Ensured price, cost, and date fields were converted to appropriate numeric and datetime formats.
- Cross-validated aggregations by reviewing sample records and summary statistics.

---

### Key Assumptions
- Sales transactions accurately represent customer demand.
- Product prices remain consistent over the observed period.
- Store performance differences are primarily demand-driven rather than operational constraints.
- Inventory availability is assumed to be sufficient unless demand signals indicate otherwise.

---

### Limitations
- The dataset does not include explicit inventory stock levels or replenishment cycles.
- Customer-level data is unavailable, limiting customer segmentation analysis.
- External factors such as promotions, holidays, or supply disruptions are not captured.
- Findings are based on historical data and may not reflect future demand patterns.

---

### Future Enhancements
- Incorporate inventory stock and replenishment data to enable stock-out and overstock analysis.
- Add promotion and pricing change data to assess demand sensitivity.
- Extend analysis to customer-level behavior if data becomes available.

In [48]:
# Export cleaned & aggregated tables for Tableau

product_performance.to_csv("product_performance.csv", index=False)
monthly_trend.to_csv("monthly_trend.csv", index=False)
store_performance.to_csv("store_performance.csv", index=False)
top_product_per_store.to_csv("top_product_per_store.csv", index=False)